In [1]:
import socket
import threading
import time
from collections import deque
import ipywidgets as widgets
from IPython.display import display

# Konfiguration
HOST = '0.0.0.0'
PORT = 5000

# Globale Variablen für Thread-Sicherheit
state_lock = threading.Lock()
running = True  # Um Threads sauber zu stoppen
queue = deque()
current_holder = None
known_nodes = set()

In [2]:
# Widgets erstellen
header_widget = widgets.HTML(value="<h3>Coordinator</h3>")
status_widget = widgets.HTML(value="Warte auf Nodes...")
stop_button = widgets.Button(description="Server Stoppen", button_style='danger')

def update_dashboard():
    """Baut die HTML-Anzeige basierend auf dem aktuellen Status"""
    while running:
        with state_lock:
            # Eigene IP ermitteln
            my_ip = socket.gethostbyname(socket.gethostname())

            # HTML bauen
            html = f"<b>Server IP:</b> {my_ip} | <b>Port:</b> {PORT}<br><hr>"
            html += "<div style='display: flex; gap: 10px; flex-wrap: wrap;'>"

            waiting_ids = [x[0] for x in queue]

            for nid in sorted(known_nodes):
                color = "#7f8c8d" # Grau (Idle)
                border = "2px solid #95a5a6"
                status_text = "IDLE"
                box_shadow = "none"
                transform = "scale(1.0)"

                if nid == current_holder:
                    color = "#27ae60" # Grün (Critical)
                    border = "2px solid #2ecc71"
                    status_text = "<b>CRITICAL SECTION</b>"
                    box_shadow = "0 0 10px #2ecc71"
                    transform = "scale(1.1)"
                elif nid in waiting_ids:
                    pos = waiting_ids.index(nid) + 1
                    color = "#f39c12" # Orange (Waiting)
                    border = "2px solid #e67e22"
                    status_text = f"Wartet (Pos {pos})"

                # CSS Karte für den Node
                card = f"""
                <div style="
                    background-color: {color};
                    border: {border};
                    padding: 15px;
                    border-radius: 8px;
                    width: 120px;
                    color: white;
                    text-align: center;
                    font-family: sans-serif;
                    box-shadow: {box_shadow};
                    transition: all 0.2s;
                    transform: {transform};
                ">
                    <div style="font-size: 1.2em; margin-bottom: 5px;">{nid}</div>
                    <div style="font-size: 0.8em;">{status_text}</div>
                </div>
                """
                html += card

            html += "</div>"

            if not known_nodes:
                html += "<p style='color: #7f8c8d;'><i>Noch keine Nodes verbunden... Starte die node.py Skripte auf deinen Geräten!</i></p>"

            status_widget.value = html

        time.sleep(0.2) # Update Rate (5 FPS)

def on_stop_click(b):
    global running
    running = False
    print("Server wird gestoppt...")

stop_button.on_click(on_stop_click)

In [3]:
def udp_server():
    global current_holder, running

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.bind((HOST, PORT))
    sock.settimeout(1.0) # Timeout damit wir die Schleife abbrechen können

    print(f"UDP Listener gestartet auf Port {PORT}")

    while running:
        try:
            try:
                data, addr = sock.recvfrom(1024)
            except socket.timeout:
                continue # Prüfen ob running noch True ist

            parts = data.decode().split()
            if len(parts) < 2: continue

            cmd, node_id = parts[0], parts[1]

            with state_lock:
                known_nodes.add(node_id)

                if cmd == 'REQ':
                    if current_holder is None:
                        current_holder = node_id
                        sock.sendto(b'GRANT', addr)
                    else:
                        queue.append((node_id, addr))

                elif cmd == 'REL':
                    if queue:
                        next_id, next_addr = queue.popleft()
                        current_holder = next_id
                        sock.sendto(b'GRANT', next_addr)
                    else:
                        current_holder = None

        except Exception as e:
            if running: print(f"Error: {e}")

    sock.close()
    print("UDP Socket geschlossen.")

In [4]:
running = True
known_nodes.clear()
queue.clear()
current_holder = None

# Threads starten
t_udp = threading.Thread(target=udp_server, daemon=True)
t_gui = threading.Thread(target=update_dashboard, daemon=True)

t_udp.start()
t_gui.start()

# Anzeige im Notebook
display(header_widget, stop_button, status_widget)

UDP Listener gestartet auf Port 5000

HTML(value='<h3>Coordinator</h3>')

Button(button_style='danger', description='Server Stoppen', style=ButtonStyle())

HTML(value="<b>Server IP:</b> 192.168.1.101 | <b>Port:</b> 5000<br><hr><div style='display: flex; gap: 10px; f…